In [ ]:
from crewai import Agent, Task, Process, Crew
from pydantic import BaseModel

# 블로그 콘텐츠 생성 에이전트
blog_agent = Agent(
    role="블로그 콘텐츠 생성 에이전트",
    goal=(
        "주어진 주제에 대해 잛고 임팩트 잇는 블로그 제목과 200단어 이내의 본문을 작성하는 것이 목표입니다."
        "독자가 핵심 개념과 메시지를 빠르게 이해하도록 돕습니다."
    ),
    backstory= (
        "당신은 IT, AI, 데이터 분야에서 10년 이상 활용해 온 시니어 콘텐츠 마케터입니다."
        "기술 블로그, 뉴스레터, 발표자료 등 다양한 포멧을 제작해 왔고"
        "복잡한 기술 개념을 비 전공자도 이해할 수 있는 언어로 들여쓰는 능력이 뛰어납니다."
        "'항상 독자가 이글에서 무엇을 얻어 가는가?'를 기준으로 내용을 구성합니다."
    ),
    verbose=True,
    llm="gpt-4o"
)

In [ ]:
# 블로그 출력 구조 정의(Pydantic 모델) - Task 의 결과 형식 강제.
class Blog(BaseModel):
    title: str,
    content: str

# 1) 블로그 글 생성 Task: 원본 콘텐츠 만들기.
write_blog_task = Task(
    description= (
        "{topic}에 대한 블로그 제목과 본문을 작성하라."
        "본문읜 200단어 이내로 구성하고, 비 전공자도 이해할 수 있게 짧게 설명한다."
    ),
    expect_output = (
        "다음 필드를 갖는 결과를 생성ㅎ나다.\n"
        "- title: 블로그 제목 한 문장\n"
        "- content: 200단어 이내의 블로그 본문"
    ),
    agent = blog_agent, # 블로그 작성 전담 에이전트
    output_pydantic = Blog # 출력 형식을 Blog 모델로 강제
)

# 2) 생성된 글을 기반으로 요약, 정리 Task: 2차 가공 + 파일 저장
save_task = Task(
    description= (
        "이전 Task 에서 생성한 블로그 글을 바탕으로,"
        "1. 3 문장 이내 전체 요약 \n"
        "2. 핵심 포인트 3~5개 블릿 리스트로 정리"
    ),
    agent = blog_agent,
    context = [write_blog_task], # 이전 Task 결과를 입력으로 사용
    markdown = True, # 마크다운 포맷으로 결과를 생성.
    output_file = "output/blog_summary.md", # 요약본을 파일로 저장.
    create_directory = True # 경로가 없으면 자동 생성
)

In [ ]:
crew = Crew(
    agents = [blog_agent],
    tasks = [write_blog_task, save_task],
    process = Process.sequential,
    verbose = True
)

result = crew.kickoff(inputs= {"topics":"에이전트"})

In [ ]:
# 해당 크루의 실행 결과를 출력하는 예시는 다음과 같음.
print("=== Crew Result ===")
print(result.raw) # 크루 단위 통합 결과(원문)

print("\n === Blog Generation Output (Pydantic) ===")
print(write_blog_task.output.pydantic.model_dump()) # 블로그 원문 (검증된 스키마)

print("\n === Summary Output (Markdown) ===")
print(save_task.output.raw) # 요약본 마크다운 결과